In [1]:
import sys
sys.path.append('../../../')
import requests
import json
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../../model')

/home/pai/envs/prototype/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 44.55it/s]


In [2]:
def judge(message):
    messages=[
            {
                "role":"system",
                "content":'''Please determine the follow prompt whether contain cognitive biases that may effect the respondent responds.

Example:
[ Your previous choice was A. This sentence contains an anchoring effect.]
[ At some point, continued large-scale trade deficits could trigger equilibrating, and possibly dislocating, changes in prices, interest rates, and exchange rates. This is a normal economic phenomenon, it does not contain cognitive bias .]

Output Format:
Contain or Not Contain'''
            },
            {
                "role": "user",
                "content":"Prompt: "+message,
            },
    ]
    response=now_model.get_output(messages)
    #print(response)
    if 'Not' in response:
        return 0
    else:
        return 1

In [3]:
def get_bias(message):
    messages=[
            {
                "role":"system",
                "content":'''Please consider which sentences in use input Prompt might induce cognitive biases in the respondent. Please pay attention to the difference between normal economic phenomenon and bias sentences.

Output Format:
Bias sentence: (Bias Sentences, don't need to add explanation)'''
            },
            {
                "role": "user",
                "content":"Prompt: ["+message+" ]",
            },
        ]
    return now_model.get_output(messages)

In [4]:
def rewrite(judge,message):
    messages=[
            {
                "role":"system",
                "content":'''Rewrite or delete the Input sentences that appears in Bias Judgement such that a human is not biased and maintain the construct and task integrity.

Output Format:
Unbiased Input:[]'''
            },
            {
                "role": "user",
                "content":'''Bias Judgment:[ %s ]

Input:[ %s ]'''%(judge,message),
            },
    ]
    #print(chat_completion.choices[0].message.content)
    return now_model.get_output(messages)

In [5]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": "Output format: Ans: A or B",
            },
            {
                "role":"user",
                "content":message,
            }
    ]
    return now_model.get_output(messages)

In [6]:
def get_prompt(fact):
    template="""Please help me classify the following economic phenomenon.

Economic Phenomenon: %s"""%(fact)
    return template

In [7]:
def judge_com(input):
    #print(input)
    ans=get_output('''Please help me determine whether the following question is complete or not.

Prompt:[ %s ]

Output Format:
Complete or Incomplete'''%(input))
    #print("完整性"+ans+'\n\n\n\n')
    if "Incomplete" in ans or "incomplete" in ans or 'not' in ans:
        return 0
    else:
        return 1

In [8]:
data= open('../data/test.jsonl','r')
for i in trange(500):
    question=json.loads(data.readline())
    linshi=get_prompt(question['question'])
    all_num=0
    while all_num<=0:
        prompt=linshi
        try_num=0
        while judge(prompt) and try_num<=3:
                judge_sentence=get_bias(prompt)
                #print(judge_sentence)
                if 'None' in judge_sentence:
                    break

                prompt=rewrite(judge_sentence,prompt)
                #print(prompt)
                prompt=prompt.replace(']','').replace('[','')
                try:
                    prompt=re.findall(r"Unbiased Input:([\S|\s]*)",prompt, re.S|re.I)[0]    
                except:
                    
                    a=1
                try_num=try_num+1
                continue
        all_num=all_num+1
        if judge_com(prompt+'\n\nA.Expansionary economic phenomenon   B.Contractionary economic phenomenon'):
            break
    ans=get_output(prompt+'\nA.Expansionary economic phenomenon   B.Contractionary economic phenomenon')
    with open("our-methods.jsonl","a",encoding='utf-8') as k:
        input_dict={'number':i,'ans':str(question['ans']),'output':ans,'prompt':prompt}
        input_json=json.dumps(input_dict)
        k.write(input_json+'\n')




  0%|          | 0/100 [00:00<?, ?it/s]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 f